# SKU Inventory Data Integration: Public Portfolio Version

This notebook presents a structured data integration workflow for reconciling product inventory across supplier feeds and an ecommerce catalog. It is written as a client-facing walkthrough: the technical implementation is visible, but each step is connected to the operational question it supports.

The original project used private operational sources. For public portfolio use, all source systems, paths, credentials, product names, supplier names, and commercially sensitive details have been anonymized. The sample files generated in this notebook preserve the same integration patterns and data quality challenges while remaining safe to share.


## Business and Technical Context

The engagement objective was to create a dependable SKU-level inventory view that could support ecommerce availability, catalog maintenance, and operational review. The workflow brings together three types of data assets:

| Anonymized source | File pattern | Business role | Technical consideration |
|---|---:|---|---|
| `source_a_inventory.csv` | Semicolon-delimited CSV | Supplier inventory and commercial attributes | European decimal formatting, headerless export, extra operational columns |
| `source_b_inventory.txt` | Pipe-delimited TXT | Secondary supplier stock, pricing, and SKU identifiers | Fixed feed order, missing EAN values, whitespace normalization |
| `product_catalog.xml` | XML catalog export | Ecommerce catalog metadata and catalog-only products | Nested XML structure, zipped delivery format, catalog coverage validation |

The intended production process is straightforward and auditable:

1. Ingest each feed from a secure transfer or mounted storage location.
2. Detect encoding before parsing to reduce ingestion failures.
3. Standardize product identifiers, numeric fields, and descriptive attributes.
4. Merge supplier feeds on the shared product identifier.
5. Reconcile the supplier result against the ecommerce catalog.
6. Produce analytics columns that explain availability, pricing consistency, source coverage, and remediation priority.


In [1]:
from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET

import chardet
import numpy as np
import pandas as pd


## 1. Anonymized Portfolio Data Setup

For this public version, the notebook creates local demo files that mirror the structure of the private feeds. This makes the integration logic reproducible for review while separating the demonstration from confidential infrastructure and client data.

In a production deployment, this setup cell would be replaced by the secure data access layer: SFTP, object storage, mounted network volumes, or an orchestrated ingestion job.


In [2]:
RAW_DIR = Path("portfolio_data/raw")
OUTPUT_DIR = Path("output")
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

source_a_path = RAW_DIR / "source_a_inventory.csv"
source_b_path = RAW_DIR / "source_b_inventory.txt"
catalog_xml_path = RAW_DIR / "product_catalog.xml"
catalog_zip_path = RAW_DIR / "product_catalog.zip"

source_a_rows = [
    ["12345678", "Hydrating Gel 50ml", "Supplier Alpha", "12,50", "8000000000012", "9,10", "", "", 24, "", "", "", "", "", 22, "SKU-A-001"],
    ["23456789", "Vitamin C Tablets", "Supplier Alpha", "18,90", "8000000000029", "13,75", "", "", 8, "", "", "", "", "", 10, "SKU-A-002"],
    ["34567890", "Digital Thermometer", "Supplier Alpha", "21,00", "8000000000036", "16,40", "", "", 0, "", "", "", "", "", 22, "SKU-A-003"],
    ["45678901", "Sensitive Skin Cream", "Supplier Alpha", "15,30", "", "10,20", "", "", 13, "", "", "", "", "", 22, "SKU-A-004"],
    ["56789012", "Seasonal Supplement", "Supplier Alpha", "24,90", "8000000000050", "19,10", "", "", 5, "", "", "", "", "", 10, "SKU-A-005"],
]

pd.DataFrame(source_a_rows).to_csv(source_a_path, sep=";", header=False, index=False, encoding="utf-8")

source_b_lines = [
    "AIC000001|8000000000012|012345678|Supplier Beta|Hydrating Gel 50ml|19|9.40|22|12.50|SKU-B-001|Y|",
    "AIC000002|8000000000029|023456789|Supplier Beta|Vitamin C Tablets|10|13.10|10|18.90|SKU-B-002|Y|",
    "AIC000003|             |034567890|Supplier Beta|Digital Thermometer|4|16.90|22|21.00|SKU-B-003|Y|",
    "AIC000004|8000000000043|045678901|Supplier Beta|Sensitive Skin Cream|16|10.10|22|15.30|SKU-B-004|Y|",
    "AIC000006|8000000000067|067890123|Supplier Beta|Catalog Missing Item|11|7.25|22|11.90|SKU-B-006|N|",
]
source_b_path.write_text("\n".join(source_b_lines), encoding="utf-8")

root = ET.Element("Catalog")
for item in [
    {"product_id": "012345678", "aic": "AIC000001", "ean": "8000000000012", "name": "Hydrating Gel 50ml", "supplier": "Catalog Brand A", "qty": "21", "unit_price": "9.25"},
    {"product_id": "023456789", "aic": "AIC000002", "ean": "8000000000029", "name": "Vitamin C Tablets", "supplier": "Catalog Brand B", "qty": "7", "unit_price": "13.55"},
    {"product_id": "034567890", "aic": "AIC000003", "ean": "8000000000036", "name": "Digital Thermometer", "supplier": "Catalog Brand C", "qty": "2", "unit_price": "17.20"},
    {"product_id": "056789012", "aic": "AIC000005", "ean": "8000000000050", "name": "Seasonal Supplement", "supplier": "Catalog Brand D", "qty": "5", "unit_price": "19.00"},
    {"product_id": "099999999", "aic": "AIC000999", "ean": "8000000000999", "name": "Catalog Only Product", "supplier": "Catalog Brand Z", "qty": "3", "unit_price": "8.90"},
]:
    product = ET.SubElement(root, "Product")
    ET.SubElement(product, "ProductID").text = item["product_id"]
    ET.SubElement(product, "AIC").text = item["aic"]
    ET.SubElement(product, "EAN").text = item["ean"]
    ET.SubElement(product, "Name").text = item["name"]
    ET.SubElement(product, "Supplier").text = item["supplier"]
    ET.SubElement(product, "Quantity").text = item["qty"]
    ET.SubElement(product, "UnitPrice").text = item["unit_price"]

ET.ElementTree(root).write(catalog_xml_path, encoding="utf-8", xml_declaration=True)
with zipfile.ZipFile(catalog_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(catalog_xml_path, arcname="product_catalog.xml")

print(f"Created demo input files in: {RAW_DIR}")


Created demo input files in: portfolio_data/raw


## 2. Reusable Parsing and Standardization Helpers

The original exploratory work included repeated cleanup steps across files. Here, those operations are centralized into helper functions so the transformation logic is consistent, easier to review, and easier to reuse in a scheduled pipeline.

The key controls are encoding detection, product ID normalization, decimal parsing, and text cleanup. These are small steps technically, but they are important for preventing silent join failures and inconsistent downstream reporting.


In [3]:
def detect_encoding(path: Path, sample_size: int = 100_000) -> str:
    """Detect text encoding from a sample of bytes."""
    raw = path.read_bytes()[:sample_size]
    result = chardet.detect(raw)
    return result.get("encoding") or "utf-8"


def normalize_product_id(series: pd.Series, width: int = 9) -> pd.Series:
    """Standardize product IDs as zero-padded strings."""
    return series.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(width)


def parse_decimal(series: pd.Series) -> pd.Series:
    """Handle decimal commas from European CSV exports."""
    return pd.to_numeric(series.astype(str).str.replace(",", ".", regex=False), errors="coerce")


def clean_text(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().replace({"": np.nan, "nan": np.nan})


## 3. Source A Ingestion: Semicolon CSV Supplier Feed

Source A represents a supplier export delivered as a headerless semicolon-delimited CSV. The ingestion step selects only the fields required for matching, pricing, inventory, tax, and SKU analysis.

From a data infrastructure perspective, this cell demonstrates schema assignment at read time. From a business perspective, it prepares the commercial attributes needed to calculate sellable stock and compare supplier pricing.


In [4]:
source_a_encoding = detect_encoding(source_a_path)
source_a_columns = [
    "product_id", "description", "supplier", "list_price", "ean", "unit_price",
    "unused_1", "unused_2", "qty", "unused_3", "unused_4", "unused_5",
    "unused_6", "unused_7", "vat_rate", "sku"
]

source_a = pd.read_csv(
    source_a_path,
    sep=";",
    header=None,
    names=source_a_columns,
    usecols=["product_id", "description", "supplier", "list_price", "ean", "unit_price", "qty", "vat_rate", "sku"],
    encoding=source_a_encoding,
    dtype={"product_id": "string", "ean": "string", "sku": "string"},
)

source_a["product_id"] = normalize_product_id(source_a["product_id"])
source_a["description"] = clean_text(source_a["description"])
source_a["supplier"] = clean_text(source_a["supplier"])
source_a["ean"] = clean_text(source_a["ean"]).fillna("UNKNOWN")
source_a["list_price"] = parse_decimal(source_a["list_price"])
source_a["unit_price"] = parse_decimal(source_a["unit_price"])
source_a["qty"] = pd.to_numeric(source_a["qty"], errors="coerce").fillna(0).astype(int)
source_a["vat_rate"] = pd.to_numeric(source_a["vat_rate"], errors="coerce")

source_a


,product_id,description,supplier,list_price,ean,unit_price,qty,vat_rate,sku
0,012345678,Hydrating Gel 50ml,Supplier Alpha,12.5,8000000000012,9.10,24,22,SKU-A-001
1,023456789,Vitamin C Tablets,Supplier Alpha,18.9,8000000000029,13.75,8,10,SKU-A-002
2,034567890,Digital Thermometer,Supplier Alpha,21.0,8000000000036,16.40,0,22,SKU-A-003
3,045678901,Sensitive Skin Cream,Supplier Alpha,15.3,<NA>,10.20,13,22,SKU-A-004
4,056789012,Seasonal Supplement,Supplier Alpha,24.9,8000000000050,19.10,5,10,SKU-A-005


## 4. Source B Ingestion: Pipe-Delimited Supplier Feed

Source B represents a second supplier feed with a fixed pipe-delimited layout. The workflow retains the operational fields required for reconciliation: AIC, EAN, product ID, supplier, description, quantity, price, VAT rate, SKU, and feed flag.

The important client-facing point is that this source is not simply appended to Source A. It is normalized first, then used as a second opinion on price, stock, and product identity.


In [5]:
source_b_encoding = detect_encoding(source_b_path)
source_b_columns = ["aic", "ean", "product_id", "supplier", "description", "qty", "unit_price", "vat_rate", "list_price", "sku", "feed_flag", "unused"]

source_b = pd.read_csv(
    source_b_path,
    sep="|",
    header=None,
    names=source_b_columns,
    encoding=source_b_encoding,
    dtype="string",
).drop(columns="unused")

source_b["product_id"] = normalize_product_id(source_b["product_id"])
source_b["description"] = clean_text(source_b["description"])
source_b["supplier"] = clean_text(source_b["supplier"])
source_b["ean"] = clean_text(source_b["ean"]).replace({"             ": np.nan}).fillna("UNKNOWN")
source_b["qty"] = pd.to_numeric(source_b["qty"], errors="coerce").fillna(0).astype(int)
source_b["unit_price"] = pd.to_numeric(source_b["unit_price"], errors="coerce")
source_b["list_price"] = pd.to_numeric(source_b["list_price"], errors="coerce")
source_b["vat_rate"] = pd.to_numeric(source_b["vat_rate"], errors="coerce")

source_b


,aic,ean,product_id,supplier,description,qty,unit_price,vat_rate,list_price,sku,feed_flag
0,AIC000001,8000000000012,012345678,Supplier Beta,Hydrating Gel 50ml,19,9.4,22,12.5,SKU-B-001,Y
1,AIC000002,8000000000029,023456789,Supplier Beta,Vitamin C Tablets,10,13.1,10,18.9,SKU-B-002,Y
2,AIC000003,UNKNOWN,034567890,Supplier Beta,Digital Thermometer,4,16.9,22,21.0,SKU-B-003,Y
3,AIC000004,8000000000043,045678901,Supplier Beta,Sensitive Skin Cream,16,10.1,22,15.3,SKU-B-004,Y
4,AIC000006,8000000000067,067890123,Supplier Beta,Catalog Missing Item,11,7.25,22,11.9,SKU-B-006,N


## 5. Ecommerce Catalog Ingestion: Zipped XML Export

The catalog feed is delivered as XML, packaged in a zip archive. This is a common pattern for platform exports where product metadata is richer and more hierarchical than flat supplier files.

The catalog is intentionally kept separate until after the supplier reconciliation. That sequencing allows the notebook to distinguish between supplier availability issues and catalog mapping gaps.


In [6]:
with zipfile.ZipFile(catalog_zip_path) as zf:
    with zf.open("product_catalog.xml") as xml_file:
        root = ET.parse(xml_file).getroot()

catalog_records = []
for product in root.findall("Product"):
    catalog_records.append({
        "product_id": product.findtext("ProductID"),
        "aic": product.findtext("AIC"),
        "ean": product.findtext("EAN"),
        "description": product.findtext("Name"),
        "supplier": product.findtext("Supplier"),
        "catalog_qty": product.findtext("Quantity"),
        "catalog_unit_price": product.findtext("UnitPrice"),
    })

catalog = pd.DataFrame(catalog_records)
catalog["product_id"] = normalize_product_id(catalog["product_id"])
catalog["description"] = clean_text(catalog["description"])
catalog["supplier"] = clean_text(catalog["supplier"])
catalog["ean"] = clean_text(catalog["ean"])
catalog["catalog_qty"] = pd.to_numeric(catalog["catalog_qty"], errors="coerce").fillna(0).astype(int)
catalog["catalog_unit_price"] = pd.to_numeric(catalog["catalog_unit_price"], errors="coerce")

catalog


,product_id,aic,ean,description,supplier,catalog_qty,catalog_unit_price
0,012345678,AIC000001,8000000000012,Hydrating Gel 50ml,Catalog Brand A,21,9.25
1,023456789,AIC000002,8000000000029,Vitamin C Tablets,Catalog Brand B,7,13.55
2,034567890,AIC000003,8000000000036,Digital Thermometer,Catalog Brand C,2,17.20
3,056789012,AIC000005,8000000000050,Seasonal Supplement,Catalog Brand D,5,19.00
4,099999999,AIC000999,8000000000999,Catalog Only Product,Catalog Brand Z,3,8.90


## 6. Pre-Merge Data Quality Controls

Before any merge, the workflow checks for duplicate product identifiers, missing keys, and invalid negative values. These controls are included because integration errors are often introduced before the business logic runs.

For a client implementation, this section would typically become part of a data quality gate: if duplicate keys or impossible commercial values appear, the pipeline can fail early or route exceptions for review.


In [7]:
def quality_report(df: pd.DataFrame, source_name: str, key: str = "product_id") -> dict:
    numeric_columns = [column for column in ["qty", "unit_price", "list_price", "catalog_qty", "catalog_unit_price"] if column in df.columns]
    negative_values = int(sum((df[column] < 0).sum() for column in numeric_columns))
    return {
        "source": source_name,
        "rows": len(df),
        "duplicate_product_ids": int(df.duplicated(key).sum()),
        "missing_product_ids": int(df[key].isna().sum()),
        "negative_numeric_values": negative_values,
    }

quality_summary = pd.DataFrame([
    quality_report(source_a, "Source A"),
    quality_report(source_b, "Source B"),
    quality_report(catalog, "Catalog"),
])
quality_summary


,source,rows,duplicate_product_ids,missing_product_ids,negative_numeric_values
0,Source A,5,0,0,0
1,Source B,5,0,0,0
2,Catalog,5,0,0,0


## 7. Supplier Feed Reconciliation

The supplier feeds are merged on the standardized product ID. The reconciliation rule used in the exploratory notebook is retained because it reflects a conservative ecommerce operating model:

- Select the higher unit price when supplier prices disagree.
- Select the lower quantity when supplier stock counts disagree.

The result is a safer availability view. It avoids overstating inventory and provides clear diagnostic columns showing which source constrained price or stock.


In [8]:
supplier_merge = pd.merge(
    source_b,
    source_a,
    on="product_id",
    how="inner",
    suffixes=("_b", "_a"),
    validate="one_to_one",
)

supplier_merge["selected_unit_price"] = supplier_merge[["unit_price_b", "unit_price_a"]].max(axis=1)
supplier_merge["sellable_qty"] = supplier_merge[["qty_b", "qty_a"]].min(axis=1)
supplier_merge["selected_price_source"] = np.where(
    supplier_merge["unit_price_b"] >= supplier_merge["unit_price_a"], "Source B", "Source A"
)
supplier_merge["stock_constraint_source"] = np.where(
    supplier_merge["qty_b"] <= supplier_merge["qty_a"], "Source B", "Source A"
)

supplier_merge["price_gap"] = (supplier_merge["unit_price_b"] - supplier_merge["unit_price_a"]).round(2)
supplier_merge["price_gap_pct"] = np.where(
    supplier_merge[["unit_price_b", "unit_price_a"]].min(axis=1) > 0,
    supplier_merge["price_gap"].abs() / supplier_merge[["unit_price_b", "unit_price_a"]].min(axis=1),
    np.nan,
).round(4)
supplier_merge["stock_gap"] = supplier_merge["qty_b"] - supplier_merge["qty_a"]

supplier_view = supplier_merge[[
    "product_id", "aic", "ean_b", "sku_b", "description_b", "supplier_b",
    "selected_unit_price", "sellable_qty", "selected_price_source", "stock_constraint_source",
    "price_gap", "price_gap_pct", "stock_gap",
]].rename(columns={
    "ean_b": "ean",
    "sku_b": "sku",
    "description_b": "description",
    "supplier_b": "supplier",
})

supplier_view


,product_id,aic,ean,sku,description,supplier,selected_unit_price,sellable_qty,selected_price_source,stock_constraint_source,price_gap,price_gap_pct,stock_gap
0,012345678,AIC000001,8000000000012,SKU-B-001,Hydrating Gel 50ml,Supplier Beta,9.4,19,Source B,Source B,0.3,0.0330,-5
1,023456789,AIC000002,8000000000029,SKU-B-002,Vitamin C Tablets,Supplier Beta,13.75,8,Source A,Source A,-0.65,0.0496,2
2,034567890,AIC000003,UNKNOWN,SKU-B-003,Digital Thermometer,Supplier Beta,16.9,0,Source B,Source A,0.5,0.0305,4
3,045678901,AIC000004,8000000000043,SKU-B-004,Sensitive Skin Cream,Supplier Beta,10.2,13,Source A,Source A,-0.1,0.0099,3


## 8. Supplier-to-Catalog Reconciliation

After supplier reconciliation, the integrated supplier view is merged with the ecommerce catalog using an outer join. This is intentional: the business needs to see not only matched products, but also gaps on either side.

The merge indicator is converted into a coverage classification so category, operations, and data teams can quickly separate matched products from supplier-only and catalog-only records.


In [9]:
merged_catalog = pd.merge(
    supplier_view,
    catalog,
    on="product_id",
    how="outer",
    suffixes=("_supplier", "_catalog"),
    indicator=True,
    validate="one_to_one",
)

merged_catalog["aic"] = merged_catalog["aic_supplier"].combine_first(merged_catalog["aic_catalog"])
merged_catalog["ean"] = merged_catalog["ean_supplier"].combine_first(merged_catalog["ean_catalog"])
merged_catalog["sku"] = merged_catalog["sku"].fillna("CATALOG_ONLY")
merged_catalog["description"] = merged_catalog["description_supplier"].combine_first(merged_catalog["description_catalog"])
merged_catalog["supplier"] = merged_catalog["supplier_supplier"].combine_first(merged_catalog["supplier_catalog"])
merged_catalog["selected_unit_price"] = merged_catalog["selected_unit_price"].combine_first(merged_catalog["catalog_unit_price"])
merged_catalog["sellable_qty"] = merged_catalog["sellable_qty"].combine_first(merged_catalog["catalog_qty"]).fillna(0).astype(int)

merged_catalog["source_coverage"] = merged_catalog["_merge"].map({
    "both": "supplier_and_catalog",
    "left_only": "supplier_only",
    "right_only": "catalog_only",
})

base_columns = [
    "product_id", "aic", "ean", "sku", "description", "supplier",
    "selected_unit_price", "sellable_qty", "source_coverage",
    "selected_price_source", "stock_constraint_source", "price_gap", "price_gap_pct", "stock_gap",
    "catalog_qty", "catalog_unit_price",
]
merged_catalog = merged_catalog[base_columns].sort_values("product_id").reset_index(drop=True)

merged_catalog


,product_id,aic,ean,sku,description,supplier,selected_unit_price,sellable_qty,source_coverage,selected_price_source,stock_constraint_source,price_gap,price_gap_pct,stock_gap,catalog_qty,catalog_unit_price
0,012345678,AIC000001,8000000000012,SKU-B-001,Hydrating Gel 50ml,Supplier Beta,9.4,19,supplier_and_catalog,Source B,Source B,0.3,0.0330,-5.0,21.0,9.25
1,023456789,AIC000002,8000000000029,SKU-B-002,Vitamin C Tablets,Supplier Beta,13.75,8,supplier_and_catalog,Source A,Source A,-0.65,0.0496,2.0,7.0,13.55
2,034567890,AIC000003,UNKNOWN,SKU-B-003,Digital Thermometer,Supplier Beta,16.9,0,supplier_and_catalog,Source B,Source A,0.5,0.0305,4.0,2.0,17.20
3,045678901,AIC000004,8000000000043,SKU-B-004,Sensitive Skin Cream,Supplier Beta,10.2,13,supplier_only,Source A,Source A,-0.1,0.0099,3.0,NaN,NaN
4,056789012,AIC000005,8000000000050,CATALOG_ONLY,Seasonal Supplement,Catalog Brand D,19.0,5,catalog_only,NaN,NaN,<NA>,NaN,NaN,5.0,19.00
5,099999999,AIC000999,8000000000999,CATALOG_ONLY,Catalog Only Product,Catalog Brand Z,8.9,3,catalog_only,NaN,NaN,<NA>,NaN,NaN,3.0,8.90


## 9. Business Analytics Layer

This section turns the integrated dataset into decision-ready fields. The analytics columns are designed to explain both the operational state of each SKU and the reason it may require action.

| Column | Client-facing interpretation |
|---|---|
| `selected_unit_price` | Commercial price selected after applying the conservative supplier rule |
| `sellable_qty` | Quantity available for ecommerce after applying the lower-stock rule |
| `source_coverage` | Whether the SKU is present in suppliers, catalog, or both |
| `price_gap_pct` | Relative supplier pricing variance for commercial review |
| `stock_gap` | Supplier stock disagreement for operational review |
| `availability_status` | In-stock, low-stock, or out-of-stock status for business users |
| `catalog_alignment_status` | Catalog mapping state for remediation planning |
| `sellable_inventory_value` | Estimated value of sellable units currently available |
| `review_priority` | Rule-based triage category for follow-up action |


In [10]:
analytics = merged_catalog.copy()
analytics["availability_status"] = np.select(
    [analytics["sellable_qty"].eq(0), analytics["sellable_qty"].between(1, 5), analytics["sellable_qty"].gt(5)],
    ["out_of_stock", "low_stock", "in_stock"],
    default="unknown",
)

analytics["catalog_alignment_status"] = analytics["source_coverage"].map({
    "supplier_and_catalog": "matched",
    "supplier_only": "missing_from_catalog",
    "catalog_only": "missing_from_supplier_feeds",
})

analytics["sellable_inventory_value"] = (analytics["selected_unit_price"] * analytics["sellable_qty"]).round(2)
analytics["price_gap_flag"] = analytics["price_gap_pct"].fillna(0).gt(0.05)
analytics["stock_gap_flag"] = analytics["stock_gap"].fillna(0).abs().gt(5)
analytics["catalog_gap_flag"] = analytics["source_coverage"].ne("supplier_and_catalog")

analytics["review_priority"] = np.select(
    [
        analytics["catalog_gap_flag"],
        analytics["availability_status"].eq("out_of_stock"),
        analytics["price_gap_flag"] | analytics["stock_gap_flag"],
    ],
    ["catalog_mapping", "stock_replenishment", "commercial_review"],
    default="no_action",
)

analytics


,product_id,aic,ean,sku,description,supplier,selected_unit_price,sellable_qty,source_coverage,selected_price_source,...,stock_gap,catalog_qty,catalog_unit_price,availability_status,catalog_alignment_status,sellable_inventory_value,price_gap_flag,stock_gap_flag,catalog_gap_flag,review_priority
0,012345678,AIC000001,8000000000012,SKU-B-001,Hydrating Gel 50ml,Supplier Beta,9.4,19,supplier_and_catalog,Source B,...,-5.0,21.0,9.25,in_stock,matched,178.6,False,False,False,no_action
1,023456789,AIC000002,8000000000029,SKU-B-002,Vitamin C Tablets,Supplier Beta,13.75,8,supplier_and_catalog,Source A,...,2.0,7.0,13.55,in_stock,matched,110.0,False,False,False,no_action
2,034567890,AIC000003,UNKNOWN,SKU-B-003,Digital Thermometer,Supplier Beta,16.9,0,supplier_and_catalog,Source B,...,4.0,2.0,17.20,out_of_stock,matched,0.0,False,False,False,stock_replenishment
3,045678901,AIC000004,8000000000043,SKU-B-004,Sensitive Skin Cream,Supplier Beta,10.2,13,supplier_only,Source A,...,3.0,NaN,NaN,in_stock,missing_from_catalog,132.6,False,False,True,catalog_mapping
4,056789012,AIC000005,8000000000050,CATALOG_ONLY,Seasonal Supplement,Catalog Brand D,19.0,5,catalog_only,NaN,...,NaN,5.0,19.00,low_stock,missing_from_supplier_feeds,95.0,False,False,True,catalog_mapping
5,099999999,AIC000999,8000000000999,CATALOG_ONLY,Catalog Only Product,Catalog Brand Z,8.9,3,catalog_only,NaN,...,NaN,3.0,8.90,low_stock,missing_from_supplier_feeds,26.7,False,False,True,catalog_mapping


## 10. Management Summary Tables

The summary tables aggregate the SKU-level output into views that are suitable for client review. They quantify catalog coverage, availability, review workload, and inventory value.

These summaries are intentionally simple: they are designed to validate the integration logic first, then provide a foundation for dashboards, exception reports, or scheduled monitoring.


In [11]:
coverage_summary = (
    analytics.groupby(["source_coverage", "catalog_alignment_status"], dropna=False)
    .agg(products=("product_id", "count"), sellable_qty=("sellable_qty", "sum"), inventory_value=("sellable_inventory_value", "sum"))
    .reset_index()
    .sort_values("products", ascending=False)
)

availability_summary = (
    analytics.groupby(["availability_status", "review_priority"], dropna=False)
    .agg(products=("product_id", "count"), inventory_value=("sellable_inventory_value", "sum"))
    .reset_index()
    .sort_values(["review_priority", "products"], ascending=[True, False])
)

display(coverage_summary)
display(availability_summary)


/var/folders/kw/95mxxlw14sv5p_lrc51kvxsc0000gn/T/ipykernel_28583/763980094.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analytics.groupby(["source_coverage", "catalog_alignment_status"], dropna=False)


,source_coverage,catalog_alignment_status,products,sellable_qty,inventory_value
8,supplier_and_catalog,matched,3,27,288.6
4,catalog_only,missing_from_supplier_feeds,2,8,121.7
0,supplier_only,missing_from_catalog,1,13,132.6
1,supplier_only,missing_from_supplier_feeds,0,0,0.0
2,supplier_only,matched,0,0,0.0
3,catalog_only,missing_from_catalog,0,0,0.0
5,catalog_only,matched,0,0,0.0
6,supplier_and_catalog,missing_from_catalog,0,0,0.0
7,supplier_and_catalog,missing_from_supplier_feeds,0,0,0.0


,availability_status,review_priority,products,inventory_value
2,low_stock,catalog_mapping,2,121.7
0,in_stock,catalog_mapping,1,132.6
1,in_stock,no_action,2,288.6
3,out_of_stock,stock_replenishment,1,0.0


## 11. Clean Export for Downstream Use

The final export contains the anonymized, business-ready fields required for reporting or operational handoff. It excludes intermediate parsing columns and keeps the output focused on product identity, availability, source coverage, commercial differences, and action priority.

In production, this file could be written to a reporting database, object storage bucket, shared operations folder, or BI ingestion layer.


In [12]:
export_columns = [
    "product_id", "aic", "ean", "sku", "description", "supplier",
    "selected_unit_price", "sellable_qty", "sellable_inventory_value",
    "availability_status", "source_coverage", "catalog_alignment_status", "review_priority",
    "selected_price_source", "stock_constraint_source", "price_gap", "price_gap_pct", "stock_gap",
]

export_path = OUTPUT_DIR / "portfolio_merged_sku_inventory.csv"
analytics[export_columns].to_csv(export_path, index=False)
print(f"Exported anonymized portfolio dataset to: {export_path}")


Exported anonymized portfolio dataset to: output/portfolio_merged_sku_inventory.csv


## 12. Client Takeaways

This notebook demonstrates how an exploratory SKU integration exercise can be translated into a reproducible, auditable workflow:

- Multiple feed formats are ingested consistently: CSV, TXT, and zipped XML.
- Encoding, identifiers, decimals, and text fields are standardized before joining.
- Merge validation reduces the risk of accidental many-to-many joins.
- Supplier conflicts are resolved with conservative price and inventory rules.
- Catalog coverage is made explicit for business remediation.
- The final dataset supports operational review, ecommerce availability, and dashboarding.

The same design can be connected to secure production sources and scheduled as a repeatable data quality and inventory integration process.
